# Box Detection Results Visualization

이 노트북은 Isaac Sim에서 감지된 박스 검출 결과를 3D로 시각화합니다.
- 박스의 8개 코너 좌표 (윗면 4개 + 밑면 4개)
- 포인트 클라우드 데이터 (PLY 파일)
- 좌표계: m0609_base_link (로봇 기준)

## 1. Import Required Libraries

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from pathlib import Path
import plotly.graph_objects as go
import struct

print("All libraries imported successfully!")

## 2. Load and Parse JSON Data

In [ ]:
# JSON 파일 경로
json_path = Path("results/crate_demo/runs/20260722_184707_35scan/20260722_184707_table_boxes_filtered.json")

# JSON 데이터 로드
with open(json_path, 'r') as f:
    data = json.load(f)

# 메타데이터 출력
print(f"Coordinate Frame: {data['coordinate_frame']}")
print(f"Unit: {data['unit']}")
print(f"Box Count: {data['box_count']}")
print(f"Total Completed Points: {data['total_completed_point_count']}")
print(f"Completed PLY File: {data['completed_ply_file']}")

# 박스 데이터 확인
for i, box in enumerate(data['boxes']):
    print(f"\nBox {i}:")
    print(f"  ID: {box['box_id']}")
    print(f"  Support Type: {box['support_type']}")
    print(f"  Corner Count: {len(box['corners_m'])}")

## 3. Extract Box Coordinates

In [ ]:
# 첫 번째 박스의 코너 추출
box = data['boxes'][0]
corners = np.array(box['corners_m'])

# 윗면 (top 0-3) 과 밑면 (bottom 4-7) 분리
top_corners = corners[0:4]
bottom_corners = corners[4:8]

print("Top Corners (윗면):")
print(top_corners)
print("\nBottom Corners (밑면):")
print(bottom_corners)

# 박스의 차원 계산
width = np.linalg.norm(top_corners[0] - top_corners[1])
depth = np.linalg.norm(top_corners[1] - top_corners[2])
height = np.linalg.norm(top_corners[0] - bottom_corners[0])

print(f"\nBox Dimensions:")
print(f"  Width: {width:.3f} m")
print(f"  Depth: {depth:.3f} m")
print(f"  Height: {height:.3f} m")

## 4. Visualize 3D Box with Matplotlib

In [ ]:
fig = plt.figure(figsize=(12, 5))

# 첫 번째 플롯: 산점도
ax1 = fig.add_subplot(121, projection='3d')

# 윗면과 밑면 점들을 다른 색으로 표시
ax1.scatter(top_corners[:, 0], top_corners[:, 1], top_corners[:, 2], 
           c='red', s=100, label='Top Corners', marker='o')
ax1.scatter(bottom_corners[:, 0], bottom_corners[:, 1], bottom_corners[:, 2], 
           c='blue', s=100, label='Bottom Corners', marker='s')

# 박스 모서리 그리기 (top square)
for i in range(4):
    p1 = top_corners[i]
    p2 = top_corners[(i+1) % 4]
    ax1.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]], 'g-', linewidth=2)

# 박스 모서리 그리기 (bottom square)
for i in range(4):
    p1 = bottom_corners[i]
    p2 = bottom_corners[(i+1) % 4]
    ax1.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]], 'g-', linewidth=2)

# 수직 모서리 연결
for i in range(4):
    p1 = top_corners[i]
    p2 = bottom_corners[i]
    ax1.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]], 'g--', linewidth=1)

ax1.set_xlabel('X (m)')
ax1.set_ylabel('Y (m)')
ax1.set_zlabel('Z (m)')
ax1.set_title('3D Box Detection Results\n(Robot Base Frame: m0609_base_link)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 두 번째 플롯: XY 평면 (위에서 본 뷰)
ax2 = fig.add_subplot(122)
ax2.scatter(top_corners[:, 0], top_corners[:, 1], c='red', s=100, label='Top', marker='o')
ax2.scatter(bottom_corners[:, 0], bottom_corners[:, 1], c='blue', s=100, label='Bottom', marker='s')

# XY 평면의 모서리 그리기 (윗면)
for i in range(4):
    p1 = top_corners[i]
    p2 = top_corners[(i+1) % 4]
    ax2.plot([p1[0], p2[0]], [p1[1], p2[1]], 'r-', linewidth=2, label='Top edge' if i == 0 else '')

# XY 평면의 모서리 그리기 (밑면)
for i in range(4):
    p1 = bottom_corners[i]
    p2 = bottom_corners[(i+1) % 4]
    ax2.plot([p1[0], p2[0]], [p1[1], p2[1]], 'b--', linewidth=2, label='Bottom edge' if i == 0 else '')

ax2.set_xlabel('X (m)')
ax2.set_ylabel('Y (m)')
ax2.set_title('Top-Down View (XY Plane)')
ax2.grid(True, alpha=0.3)
ax2.axis('equal')
ax2.legend()

plt.tight_layout()
plt.show()

print("3D Box Visualization Complete!")

## 5. Visualize Point Cloud Data (PLY File)

In [ ]:
def read_ply_simple(ply_path):
    """PLY 파일에서 포인트 클라우드 좌표 추출 (간단한 구현)"""
    points = []
    with open(ply_path, 'r') as f:
        # 헤더 읽기
        line = f.readline()
        vertex_count = 0
        while line.strip() != 'end_header':
            if line.startswith('element vertex'):
                vertex_count = int(line.split()[-1])
            line = f.readline()
        
        # 포인트 데이터 읽기
        for _ in range(vertex_count):
            parts = f.readline().split()
            x, y, z = float(parts[0]), float(parts[1]), float(parts[2])
            points.append([x, y, z])
    
    return np.array(points)

# PLY 파일 경로
ply_dir = Path("results/crate_demo/runs/20260722_184707_35scan")
ply_file = data['completed_ply_file']
ply_path = ply_dir / ply_file

print(f"Loading PLY file: {ply_path}")
try:
    if ply_path.exists():
        point_cloud = read_ply_simple(ply_path)
        print(f"Loaded {len(point_cloud)} points")
        print(f"Point Cloud Shape: {point_cloud.shape}")
        print(f"X range: {point_cloud[:, 0].min():.3f} to {point_cloud[:, 0].max():.3f}")
        print(f"Y range: {point_cloud[:, 1].min():.3f} to {point_cloud[:, 1].max():.3f}")
        print(f"Z range: {point_cloud[:, 2].min():.3f} to {point_cloud[:, 2].max():.3f}")
    else:
        print(f"PLY file not found at {ply_path}")
        print(f"Searched in: {ply_dir}")
        # 디렉토리 내용 확인
        if ply_dir.exists():
            print(f"Files in {ply_dir}:")
            for f in ply_dir.glob("*"):
                print(f"  - {f.name}")
        point_cloud = None
except Exception as e:
    print(f"Error loading PLY file: {e}")
    point_cloud = None

## 6. Interactive 3D Visualization with Plotly

In [ ]:
fig_plotly = go.Figure()

# 박스 코너 추가
fig_plotly.add_trace(go.Scatter3d(
    x=top_corners[:, 0],
    y=top_corners[:, 1],
    z=top_corners[:, 2],
    mode='markers+text',
    name='Top Corners',
    marker=dict(size=8, color='red', symbol='circle'),
    text=[f'T{i}' for i in range(4)],
    textposition='top center'
))

fig_plotly.add_trace(go.Scatter3d(
    x=bottom_corners[:, 0],
    y=bottom_corners[:, 1],
    z=bottom_corners[:, 2],
    mode='markers+text',
    name='Bottom Corners',
    marker=dict(size=8, color='blue', symbol='square'),
    text=[f'B{i}' for i in range(4)],
    textposition='top center'
))

# 박스 모서리 그리기
all_corners = np.vstack([top_corners, bottom_corners])

# 윗면 모서리
for i in range(4):
    p1 = top_corners[i]
    p2 = top_corners[(i+1) % 4]
    fig_plotly.add_trace(go.Scatter3d(
        x=[p1[0], p2[0]],
        y=[p1[1], p2[1]],
        z=[p1[2], p2[2]],
        mode='lines',
        line=dict(color='green', width=3),
        name='Top Edge' if i == 0 else '',
        showlegend=(i == 0),
        hoverinfo='skip'
    ))

# 밑면 모서리
for i in range(4):
    p1 = bottom_corners[i]
    p2 = bottom_corners[(i+1) % 4]
    fig_plotly.add_trace(go.Scatter3d(
        x=[p1[0], p2[0]],
        y=[p1[1], p2[1]],
        z=[p1[2], p2[2]],
        mode='lines',
        line=dict(color='green', width=3, dash='dash'),
        name='Bottom Edge' if i == 0 else '',
        showlegend=(i == 0),
        hoverinfo='skip'
    ))

# 수직 모서리
for i in range(4):
    p1 = top_corners[i]
    p2 = bottom_corners[i]
    fig_plotly.add_trace(go.Scatter3d(
        x=[p1[0], p2[0]],
        y=[p1[1], p2[1]],
        z=[p1[2], p2[2]],
        mode='lines',
        line=dict(color='gray', width=2, dash='dot'),
        name='Vertical Edge' if i == 0 else '',
        showlegend=(i == 0),
        hoverinfo='skip'
    ))

# 포인트 클라우드 추가 (있으면)
if point_cloud is not None:
    # 박스 영역의 포인트 강조
    box_start_idx = box['ply_point_start_index']
    box_end_idx = box['ply_point_end_index']
    
    if box_end_idx <= len(point_cloud):
        box_points = point_cloud[box_start_idx:box_end_idx]
        fig_plotly.add_trace(go.Scatter3d(
            x=box_points[:, 0],
            y=box_points[:, 1],
            z=box_points[:, 2],
            mode='markers',
            name='Box Point Cloud',
            marker=dict(size=2, color='yellow', opacity=0.6),
            hoverinfo='skip'
        ))

fig_plotly.update_layout(
    title='Interactive 3D Box Detection Visualization',
    scene=dict(
        xaxis_title='X (m)',
        yaxis_title='Y (m)',
        zaxis_title='Z (m)',
        aspectmode='data'
    ),
    width=1000,
    height=800,
    showlegend=True
)

fig_plotly.show()

print("Interactive visualization complete!")

## Summary

### Box Detection Results
- **Coordinate Frame**: m0609_base_link (로봇 기준좌표계)
- **Detected Boxes**: 1개
- **Box Dimensions**:
  - Width: {:.3f} m
  - Depth: {:.3f} m
  - Height: {:.3f} m

### Detected Corners
- **Top Surface**: 빨간색 원형 점 (4개)
- **Bottom Surface**: 파란색 사각형 점 (4개)
- **Green Lines**: 박스 모서리
- **Yellow Points**: 포인트 클라우드 (PLY 파일에서)

### 시각화 방법
1. **Matplotlib**: 정적 3D 플롯 (위에서 본 뷰 포함)
2. **Plotly**: 대화형 3D 시각화 (마우스로 회전/확대 가능)

이 데이터는 40번 스크립트가 생성한 박스 검출 결과입니다.